# Usage | 2. Network growth
This notebook explains how growbikenet orders the edges and why this is important.

**Parameters covered**: `ranking`, `allow_edge_overlaps`, `crs_projected`

We start every Usage notebook with the standard way of importing growbikenet:

In [ ]:
import growbikenet as gbn

## Ranking of edges

Growbikenet not only creates a potential bicycle network, but also a ranking of the edges. The ranking is important, as it provides an ordering, informing a city which edges to implement first to arrive at a functional network early. The ranking metric is controlled by the parameter `ranking` which is by default set to `'betweenness_centrality'`. Other options are `'closeness_centrality'` and `'random'`.

We are going to generate all different rankings (saved in the dictionary `edges_ranked_all`) and visualize them interactively in the end.

In [ ]:
edges_ranked_all = {}

### Betweenness centrality

Betweenness centrality is a network centrality measure approximating flow. By ordering edges like this, the first built edge is the one with highest expected flow of cyclists. As was shown in the research on which growbikenet is based, this is a much better ordering than random, and in most cases also better than ordering by closeness centrality.

In [ ]:
edges_ranked = gbn.growbikenet("Paris",
                               ranking="betweenness_centrality",)
edges_ranked_all["betweenness_centrality"] = edges_ranked

The content of the output `edges_ranked` shows several columns:

In [ ]:
edges_ranked.head()

They are:  
- **betweenness_centrality**: The metric to rank edges by.
- **geometry**: The geometries of the edges connecting source and target seed points, projected in the coordinate references system (crs) given by the parameter `crs_projected`, rounded to meters. The coordinates correspond to the easting and northing in the crs. These geometries are typically a mix between linestrings and multilinestrings. 
- **source**, **target**: The OSM IDs of source and target seed points. These are nodes that can be looked up on OSM, for example for OSM ID 11037313412: https://www.openstreetmap.org/node/11037313412
- **rank**: The rank which orders the edge by betweenness centrality. These are increasing integers, but not necessarily consecutive, due to potentially empty pieces in-between that are removed due to edge overlaps, see end of this notebook.
- **length**: Length of the current edge, rounded to whole meters.
- **length_cumulative**: Cumulative length of current and all previous edges, rounded to whole meters.

### Closeness centrality

Ranking by closeness centrality means growing from the center.

In [ ]:
edges_ranked = gbn.growbikenet("Paris",
                               ranking="closeness_centrality",)
edges_ranked_all["closeness_centrality"] = edges_ranked

### Random

Growbikenet can also showcase suboptimal random growth.

In [ ]:
edges_ranked = gbn.growbikenet("Paris",
                               ranking="random",)
edges_ranked_all["random"] = edges_ranked

### Visualization

Below the three different growths are visualized together ineractively. Use the slider and buttons to see the different growth patterns:

In [ ]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
step = widgets.IntSlider(
    value=4, min=0, max=max(edges_ranked["rank"]), step=1, description="Growth step:", layout=widgets.Layout(width='500px')
)
ranking = widgets.ToggleButtons(
    options=['betweenness_centrality', 'closeness_centrality', 'random'],
    description='Ranking:',
)
def update_map(step=step, ranking=ranking):
    fig, ax = plt.subplots(figsize=(8, 6))
    edges_ranked_all[ranking][edges_ranked_all[ranking]["rank"] <= step].plot(
        linewidth=3,
        color="#096a51",
        ax=ax,
    )
    ax.set_title(f"Paris bike net growth, growth step {step}", fontsize=12)
    ax.set_ylim(edges_ranked.total_bounds[1], edges_ranked.total_bounds[3])
    ax.set_xlim(edges_ranked.total_bounds[0], edges_ranked.total_bounds[2])
    plt.axis('off')
    plt.show()

widgets.interactive(update_map, step=step, ranking=ranking)

## Allowing edge overlaps

Considerable computing time is spent on "Removing edge overlaps". 

*What are edge overlaps?* In the abstract, unrouted network, there can be no edge overlaps in the seed network. However, when the edges are routed, there are often many such overlaps. In general, we do not want to have overlaps as they render length calculations wrong, i.e., the `length` and `length_cumulative` columns in the result:

In [ ]:
edges_ranked_all["betweenness_centrality"].head()

Because edge overlaps are removed by default, the lengths are correct. In particular, the total length of the grown network is

In [ ]:
int(edges_ranked_all["betweenness_centrality"].length_cumulative.iloc[-1]/1000)

kilometers.

Edge overlap removal also removes edges that would end up completely empty, which is the reason why the rank can show gaps. This can be seen by the difference of the following two numbers:

In [ ]:
print(len(edges_ranked_all["betweenness_centrality"]),
      max(edges_ranked_all["betweenness_centrality"]["rank"])+1)

However, if we allow edge overlaps, via `allow_edge_overlaps=True`, this will make the calculation faster:

In [ ]:
edges_ranked = gbn.growbikenet("Paris",
                               ranking="betweenness_centrality",
                               allow_edge_overlaps=True,)

It also leads to no rank gaps:

In [ ]:
print(len(edges_ranked),
      max(edges_ranked["rank"])+1)

But it will also wrongly skew the calculated total length of the network to be much longer:

In [ ]:
int(edges_ranked.length_cumulative.iloc[-1]/1000)